# Simple SWAT+ model order (USGS station)

**What you have today:** SWATGenX is an **async** service. You send a **USGS streamgage / station ID** (`site_no`, e.g. `05580950`), the API returns an **order** / **task**, and workers build the model in the background. When the run finishes, you usually get a **download link by email** (token URL) or you download from the **web dashboard** — there is **no** single HTTP call that returns the full model ZIP immediately in the JSON response.

This notebook keeps the flow **small**: configure → submit → wait until the server reports **SUCCESS** (or failure) → next steps for the file.


In [ ]:
# If needed (JupyterLab / Colab):
# !pip install -q requests


In [ ]:
import time
import json
import os

import requests

# --- edit these two lines ---
SITE_NO = os.environ.get("SWATGENX_SITE_NO", "05580950")  # USGS station ID (your "model name" for station builds)
API_KEY = os.environ.get("SWATGENX_API_KEY", "").strip()  # sgx_… from SWATGenX → Subscription / API keys
BASE_URL = os.environ.get("SWATGENX_BASE_URL", "https://www.swatgenx.com").rstrip("/")

if not API_KEY:
    raise ValueError("Set API_KEY in this cell or export SWATGENX_API_KEY before starting Jupyter.")

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "X-SWATGenX-Api-Key": API_KEY,
    "Content-Type": "application/json",
    "Accept": "application/json",
}


In [ ]:
# 1) Submit model creation (station-centered SWAT+)
payload = {
    "site_no": str(SITE_NO).strip(),
    "ls_resolution": 250,
    "dem_resolution": 30,
}
r = requests.post(f"{BASE_URL}/api/model-settings", headers=HEADERS, json=payload, timeout=120)
print("HTTP", r.status_code)
body = r.json() if r.content else {}
print(json.dumps(body, indent=2))
r.raise_for_status()
if body.get("status") != "success":
    raise RuntimeError(body)

order_id = str(body.get("order_id") or "").strip()
task_id = str(body.get("task_id") or "").strip() or None
print("order_id:", order_id, "task_id:", task_id)


In [ ]:
# 2) Poll until the Celery task finishes (or timeout)
# Prefer task_id from the submit response; otherwise resolve from /api/model-orders.

def find_task_id_for_order(oid: str) -> str | None:
    r2 = requests.get(f"{BASE_URL}/api/model-orders", headers=HEADERS, params={"limit": 50}, timeout=60)
    r2.raise_for_status()
    for row in (r2.json() or {}).get("orders") or []:
        if str(row.get("order_id")) == oid:
            tid = str(row.get("celery_task_id") or "").strip()
            return tid or None
    return None


def task_status(tid: str) -> dict:
    r3 = requests.get(f"{BASE_URL}/api/task_status/{tid}", headers=HEADERS, timeout=60)
    r3.raise_for_status()
    d = r3.json()
    if d.get("status") != "success":
        return {}
    return d.get("task_info") or {}


if not task_id and order_id:
    task_id = find_task_id_for_order(order_id)

if not task_id:
    raise RuntimeError("No task_id yet — wait a few seconds and re-run this cell, or check /api/model-orders in the app.")

deadline = time.monotonic() + 3600  # 1 h cap; shorten for tiny test stations
last = None
while time.monotonic() < deadline:
    info = task_status(task_id)
    st = str(info.get("status") or "").upper()
    last = st
    msg = (info.get("info") or {}).get("message")
    print(st, "-", msg)
    if st in ("SUCCESS", "FAILURE", "REVOKED"):
        break
    time.sleep(15)

print("Final status:", last)


## 3) Get the model files

After **SUCCESS**, SWATGenX typically emails a **secure download link** (`/download_model/<token>`). You can also sign in at [swatgenx.com](https://www.swatgenx.com) and download the workspace from your **user dashboard**.

If SWATGenX later exposes a **signed HTTPS download URL** in an API response, this notebook can be extended to `requests.get(...)` the ZIP directly — today the product flow is **order → build → email/dashboard**.
